In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

w = WorkspaceClient()
MODEL = "databricks-meta-llama-3-1-8b-instruct"

def llm(prompt):                                  # helper to call the LLM
    return w.serving_endpoints.query(
        name=MODEL,
        messages=[ChatMessage(role=ChatMessageRole.USER, content=prompt)]
    ).choices[0].message.content

# ---- TOOL 1: unstructured (reviews) ----
def search_reviews(query):
    res = w.vector_search_indexes.query_index(
        index_name="northstar.gold.reviews_index",
        columns=["review_comment_message"], query_text=query, num_results=5)
    return "\n".join(f"- {r[0]}" for r in res.result.data_array)

# ---- TOOL 2: structured (Gold table) ----
def get_late_delivery_rate():
    row = spark.sql("""SELECT ROUND(100.0*SUM(is_late)/COUNT(*),1) AS pct
                       FROM northstar.gold.fct_orders WHERE is_delivered=1""").collect()[0]
    return f"Late delivery rate: {row['pct']}%"

# ---- THE AGENT ----
def agent(question):
    # 1️⃣ DECIDE: which tool?
    choice = llm(f"""Tools available:
- search_reviews: customer opinions, complaints, feedback
- get_late_delivery_rate: delivery statistics / numbers
Question: "{question}"
Reply with ONLY the tool name.""").strip()

    # 2️⃣ ACT: call the chosen tool
    data = search_reviews(question) if "search_reviews" in choice else get_late_delivery_rate()

    # 3️⃣ ANSWER: LLM responds using the tool's result
    return llm(f"Question: {question}\n\nTool result:\n{data}\n\nAnswer concisely:")

# Test it — different questions route to different tools!
print("Q1:", agent("What do customers complain about with delivery?"))   # → search_reviews
print("Q2:", agent("What is our late delivery rate?"))                   # → get_late_delivery_rate